In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local nocodb")
else:
    print("using aws nocodb")

using aws nocodb


In [3]:
from birddog.database import Database
from birddog.runtime import Runtime
from birddog.database_updater import DatabaseUpdater
from birddog.wiki import (
    get_root_label,
    page_label,
    sequential_page_label,
    )

2026-07-06 16:01:10,931 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-07-06 16:01:10,943 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-07-06 16:01:11,144 [INFO] Translation is enabled. Using GCP translator
2026-07-06 16:01:11,145 [INFO] Using Google Cloud translation API
2026-07-06 16:01:11,145 [INFO] GoogleCloudTranslator using REST API


In [4]:
runtime = Runtime()
updater = DatabaseUpdater(runtime)

2026-07-06 16:01:11,560 [INFO] PageUpdateManager.init(): detect_environment==local
2026-07-06 16:01:12,136 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-07-06 16:01:12,249 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     8.93    39.00       0.00           24
2026-07-06 16:01:12,526 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-07-06 16:01:13,183 [INFO] WikiDocTracker: base=https://commons.wikimedia.org, namespace=File (id=6)
2026-07-06 16:01:13,371 [INFO] WikiDocTracker: base=https://uk.wikisource.org, namespace=Файл (id=6)
2026-07-06 16:01:13,373 [INFO] KillSwitch: loading thresholds from resou

In [5]:
updater.refresh_doc_lookups(limit=1000)

2026-07-06 16:01:19,460 [INFO] refresh_doc_lookups: writing 98 updates
2026-07-06 16:01:19,463 [INFO] creating Reserver(table_name=Documents)


True

In [ ]:
while updater.refresh_doc_lookups(limit=1000):
    print("finished refresh pass")

In [6]:
db = updater._db
#db = Database()

In [8]:
doc_recs, _ = db.scan(
    "Documents", 
    view_name="BD:Need Page Lookups",
    fields="url",
    limit=500,
)

2026-07-06 17:15:50,277 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   21.00     0.03    39.00       0.00           24
  uk.wikisource.org:api                  4.00     0.00     3.97       0.00            4


In [10]:
len(doc_recs)

98

In [11]:
doc_recs[0]

{'Id': 217779,
 'url': 'https://commons.wikimedia.org/wiki/File:ДАЧкО_Р-279-1-978_Справа_за_звинуваченням_Сільського_Хаїма_Лейзеровича,_громадянина_м._Корсунь,_у..._(1924).pdf'}

In [12]:
om = updater._get_owner_ids([r["Id"] for r in doc_recs])

2026-07-06 17:16:58,132 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   22.00     0.01    36.07       0.00           24
  uk.wikisource.org:api                  4.00     0.00     3.97       0.00            4


In [15]:

_lookup_fields = [
    "label",
    "seq_label",
    "root_label",
    "level",
    "description",
    "native_description",
]
_lookup_field_mapping = {
    "description": "page_description",
    "native_description": "page_native_description",
}


In [16]:
page_ids = []
for p in om.values():
    page_ids.extend(p)
page_ids = list(set(page_ids))

page_recs = db.read("Pages", page_ids, fields=_lookup_fields)

2026-07-06 17:18:50,454 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   23.00     0.87    39.00       0.00           24
  uk.wikisource.org:api                  4.00     0.00     3.97       0.00            4


In [18]:
page_map = { r["Id"]: r for r in page_recs if r }
doc_map = { r["Id"]: r for r in doc_recs }

In [19]:
up = updater._set_doc_lookup_fields(doc_map, page_map, om, _lookup_fields, _lookup_field_mapping)

In [20]:
up

[{'url': 'https://commons.wikimedia.org/wiki/File:ДАЧкО_Р-279-1-978_Справа_за_звинуваченням_Сільського_Хаїма_Лейзеровича,_громадянина_м._Корсунь,_у..._(1924).pdf',
  'lookup_status': 'valid',
  'label': None,
  'seq_label': None,
  'root_label': None,
  'level': None,
  'page_description': None,
  'page_native_description': None},
 {'url': 'https://commons.wikimedia.org/wiki/File:ДАЧкО_Р-279-1-997_Справа_про_крадіжку_речей_невідомими_злочинцями_у_Соколової_Рахілі_Нухимівни,..._(1924).pdf',
  'lookup_status': 'valid',
  'label': None,
  'seq_label': None,
  'root_label': None,
  'level': None,
  'page_description': None,
  'page_native_description': None},
 {'url': 'https://uk.wikisource.org/wiki/File:ДАЧкО_Р-279-1-978_Справа_за_звинуваченням_Сільського_Хаїма_Лейзеровича,_громадянина_м._Корсунь,_у..._(1924).pdf',
  'lookup_status': 'valid',
  'label': None,
  'seq_label': None,
  'root_label': None,
  'level': None,
  'page_description': None,
  'page_native_description': None},
 {'url'

In [ ]:
def record_update(rec):
    label = page_label(rec["title"])
    return {
        "url": rec["url"], 
        "label": label,
        "root_label": get_root_label(label),
        "seq_label": sequential_page_label(label),
    }
    
def do_pass(limit=500):
    rec, _ = db.scan(
        "Pages", 
        view_name="BD:Blank Root Label",
        fields=["url", "title"],
        limit=limit)
    if not rec:
        return False
    rec = [ record_update(r) for r in rec]
    print(f"writing {len(rec)} records")
    db.write("Pages", rec)
    return True

In [ ]:
do_pass(limit=1000)

In [ ]:
while do_pass(limit=1000):
    pass